# 01 - Data Collection with the SpaceX REST API
Requests launch data from `api.spacexdata.com/v4`, enriches it with rocket, launchpad, payload and core details, and saves a DataFrame.

In [ ]:
import requests, pandas as pd, numpy as np
pd.set_option('display.max_columns',None)
spacex_url='https://api.spacexdata.com/v4/launches/past'
response=requests.get(spacex_url)
print(response.status_code)

In [ ]:
data=pd.json_normalize(response.json())
data=data[['rocket','payloads','launchpad','cores','flight_number','date_utc']]
data=data[data['cores'].map(len)==1]
data=data[data['payloads'].map(len)==1]
data['cores']=data['cores'].map(lambda x:x[0])
data['payloads']=data['payloads'].map(lambda x:x[0])
data['date']=pd.to_datetime(data['date_utc']).dt.date
data=data[data['date']<=pd.Timestamp(2020,11,13).date()]
data.head()

In [ ]:
BoosterVersion=[];PayloadMass=[];Orbit=[];LaunchSite=[];Longitude=[];Latitude=[]
Outcome=[];Flights=[];GridFins=[];Reused=[];Legs=[];LandingPad=[];Block=[];ReusedCount=[];Serial=[]

def getBoosterVersion(d):
    for x in d['rocket']:
        if x: BoosterVersion.append(requests.get('https://api.spacexdata.com/v4/rockets/'+str(x)).json()['name'])
def getLaunchSite(d):
    for x in d['launchpad']:
        if x:
            r=requests.get('https://api.spacexdata.com/v4/launchpads/'+str(x)).json()
            Longitude.append(r['longitude']);Latitude.append(r['latitude']);LaunchSite.append(r['name'])
def getPayloadData(d):
    for load in d['payloads']:
        if load:
            r=requests.get('https://api.spacexdata.com/v4/payloads/'+load).json()
            PayloadMass.append(r['mass_kg']);Orbit.append(r['orbit'])
def getCoreData(d):
    for core in d['cores']:
        if core['core'] is not None:
            r=requests.get('https://api.spacexdata.com/v4/cores/'+core['core']).json()
            Block.append(r['block']);ReusedCount.append(r['reuse_count']);Serial.append(r['serial'])
        else:
            Block.append(None);ReusedCount.append(None);Serial.append(None)
        Outcome.append(str(core['landing_success'])+' '+str(core['landing_type']))
        Flights.append(core['flight']);GridFins.append(core['gridfins']);Reused.append(core['reused']);Legs.append(core['legs']);LandingPad.append(core['landpad'])

getBoosterVersion(data);getLaunchSite(data);getPayloadData(data);getCoreData(data)

In [ ]:
launch_dict={'FlightNumber':list(data['flight_number']),'Date':list(data['date']),'BoosterVersion':BoosterVersion,'PayloadMass':PayloadMass,'Orbit':Orbit,'LaunchSite':LaunchSite,'Outcome':Outcome,'Flights':Flights,'GridFins':GridFins,'Reused':Reused,'Legs':Legs,'LandingPad':LandingPad,'Block':Block,'ReusedCount':ReusedCount,'Serial':Serial,'Longitude':Longitude,'Latitude':Latitude}
df=pd.DataFrame(launch_dict)
data_falcon9=df[df['BoosterVersion']!='Falcon 1'].reset_index(drop=True)
data_falcon9['FlightNumber']=list(range(1,data_falcon9.shape[0]+1))
data_falcon9['PayloadMass']=data_falcon9['PayloadMass'].fillna(data_falcon9['PayloadMass'].mean())
print(data_falcon9.isnull().sum())
data_falcon9.to_csv('dataset_part_1.csv',index=False)
data_falcon9.head()

**Fallback:** if the live API is unavailable, load the IBM static copy:
`pd.read_json('https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/API_call_spacex_api.json')`